# This notebook is dedicated to Soft Actor Critic agent module development

In [16]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
import gymnasium as gym
from typing import Tuple, Dict, List
import gc
from collections import defaultdict

import matplotlib.pyplot as plt
from datetime import datetime
import json
from pathlib import Path
import logging

In [5]:
# Read and preprocess the data
df = pd.read_csv('/app/data/data/rts_1m_365d_with_indicators.csv')

# Print information about the DataFrame composition
print("\nDataFrame Info:")
print("-" * 50)
print(df.info())

print("\nDataFrame Columns:")
print("-" * 50)
print(df.columns.tolist())

print("\nSummary Statistics:")
print("-" * 50)
print(df.describe())

print("\nMissing Values:")
print("-" * 50)
print(df.isnull().sum())



DataFrame Info:
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11809037 entries, 0 to 11809036
Data columns (total 28 columns):
 #   Column          Dtype  
---  ------          -----  
 0   date            object 
 1   figi            object 
 2   Open            float64
 3   High            float64
 4   Low             float64
 5   Close           float64
 6   Volume          float64
 7   day             int64  
 8   rsi             float64
 9   macd_line       float64
 10  macd_signal     float64
 11  macd_histogram  float64
 12  sma_12          float64
 13  sma_26          float64
 14  sma_50          float64
 15  sma_100         float64
 16  atr             float64
 17  bb_lower        float64
 18  bb_middle       float64
 19  bb_upper        float64
 20  vwap            float64
 21  adx             float64
 22  adx_positive    float64
 23  adx_negative    float64
 24  stoch_k         float64
 25  stoch_d         float64
 26  

## 1.0 Feature scaler

In [6]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

class FeatureScaler:
    def __init__(self):
        # Price-based features scaler (OHLC, SMAs, BB bands, VWAP)
        self.price_scaler = RobustScaler()
        
        # Volume-based features scaler (Volume, OBV)
        self.volume_scaler = RobustScaler()
        
        # Bounded indicators scaler (RSI, Stochastic K/D)
        self.bounded_scaler = MinMaxScaler()
        
        # Unbounded indicators scaler (MACD, ATR, ADX)
        self.indicator_scaler = StandardScaler()
        
        # Define feature groups
        self.price_features = ['Open', 'High', 'Low', 'Close', 
                             'sma_12', 'sma_26', 'sma_50', 'sma_100',
                             'bb_lower', 'bb_middle', 'bb_upper', 'vwap']
        
        self.volume_features = ['Volume', 'obv']
        
        self.bounded_features = ['rsi', 'stoch_k', 'stoch_d',
                               'adx', 'adx_positive', 'adx_negative']
        
        self.indicator_features = ['macd_line', 'macd_signal', 'macd_histogram', 'atr']
        
    def fit(self, df):
        # Fit price features
        self.price_scaler.fit(df[self.price_features])
        
        # Fit volume features
        self.volume_scaler.fit(df[self.volume_features])
        
        # Fit bounded features (some are already 0-100 scaled)
        self.bounded_scaler.fit(df[self.bounded_features])
        
        # Fit unbounded indicators
        self.indicator_scaler.fit(df[self.indicator_features])
        
    def transform(self, df):
        df_scaled = df.copy()
        
        # Transform each feature group
        df_scaled[self.price_features] = self.price_scaler.transform(df[self.price_features])
        df_scaled[self.volume_features] = self.volume_scaler.transform(df[self.volume_features])
        df_scaled[self.bounded_features] = self.bounded_scaler.transform(df[self.bounded_features])
        df_scaled[self.indicator_features] = self.indicator_scaler.transform(df[self.indicator_features])
        
        return df_scaled
    
    def inverse_transform(self, df_scaled):
        df = df_scaled.copy()
        
        # Inverse transform each feature group
        df[self.price_features] = self.price_scaler.inverse_transform(df_scaled[self.price_features])
        df[self.volume_features] = self.volume_scaler.inverse_transform(df_scaled[self.volume_features])
        df[self.bounded_features] = self.bounded_scaler.inverse_transform(df_scaled[self.bounded_features])
        df[self.indicator_features] = self.indicator_scaler.inverse_transform(df_scaled[self.indicator_features])
        
        return df


## 1.1 Scaler testing

In [5]:
def analyze_scaling_results(df: pd.DataFrame, feature_scaler: FeatureScaler, n_samples: int = 1000):
    """
    Analyze and display summary statistics of original vs scaled data for each feature group
    """
    # Get a sample of data
    sample_df = df.sample(n=n_samples)
    
    # Scale the sample
    scaled_df = feature_scaler.transform(sample_df)
    
    # Print summary statistics for each feature group
    feature_groups = {
        'Price Features': feature_scaler.price_features,
        'Volume Features': feature_scaler.volume_features,
        'Bounded Features': feature_scaler.bounded_features,
        'Indicator Features': feature_scaler.indicator_features
    }
    
    for group_name, features in feature_groups.items():
        print(f"\n{group_name} Summary:")
        print("\nOriginal Data:")
        print(sample_df[features].describe())
        print("\nScaled Data:")
        print(scaled_df[features].describe())
        print("\n" + "="*80)

def test_scaling(df: pd.DataFrame):
    # Initialize scaler
    feature_scaler = FeatureScaler()
    
    # Remove non-feature columns
    feature_df = df.drop(['date', 'figi', 'tic', 'day'], axis=1)
    
    # Fit scaler
    feature_scaler.fit(feature_df)
    
    # Analyze results
    analyze_scaling_results(feature_df, feature_scaler)
    
    # Test inverse transform
    scaled_df = feature_scaler.transform(feature_df)
    inverse_df = feature_scaler.inverse_transform(scaled_df)
    
    # Check if inverse transform recovers original data
    print("\nInverse Transform Test:")
    print("Maximum absolute difference between original and inverse transformed data:")
    print((feature_df - inverse_df).abs().max().max())
    
    return feature_scaler

# Run the test
feature_scaler = test_scaling(df)


Price Features Summary:

Original Data:
               Open          High           Low         Close        sma_12  \
count   1000.000000   1000.000000   1000.000000   1000.000000   1000.000000   
mean    1252.392153   1252.738647   1251.972110   1252.408230   1252.261938   
std     2539.691591   2540.403645   2539.011586   2539.920701   2539.497753   
min        0.065040      0.065040      0.065000      0.065000      0.065028   
25%       54.235000     54.242500     54.202500     54.217500     54.216250   
50%      258.590000    258.595000    258.415000    258.440000    258.585002   
75%      891.775000    891.775000    891.775000    891.775000    892.314579   
max    14455.000000  14470.500000  14455.000000  14469.500000  14447.333333   

             sma_26        sma_50       sma_100      bb_lower     bb_middle  \
count   1000.000000   1000.000000   1000.000000   1000.000000   1000.000000   
mean    1252.263152   1252.406885   1252.755949   1249.644545   1252.258392   
std     25

## 2.0 Replay buffer


In [17]:

class MultiTickerReplayBuffer:
    """
    Replay Buffer for storing and sampling transitions with multiple tickers.
    Maintains separate storage for each ticker while allowing both ticker-specific
    and combined sampling strategies.
    """
    
    def __init__(self, capacity: int, state_dim: int, action_dim: int, tickers: List[str]):
        """
        Initialize replay buffer for multiple tickers.
        
        Args:
            capacity: Maximum number of transitions to store per ticker
            state_dim: Dimension of state space
            action_dim: Total action dimension (n_tickers * action_per_ticker)
            tickers: List of ticker symbols
        """
        self.capacity = capacity
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.tickers = tickers
        self.n_tickers = len(tickers)
        
        # Create separate storage for each ticker
        self.buffers = {
            ticker: {
                'states': np.zeros((capacity, state_dim), dtype=np.float32),
                'actions': np.zeros((capacity, self.n_tickers), dtype=np.float32),  # Changed dimension
                'rewards': np.zeros(capacity, dtype=np.float32),
                'next_states': np.zeros((capacity, state_dim), dtype=np.float32),
                'dones': np.zeros(capacity, dtype=np.float32)
            } for ticker in tickers
        }
        
        # Track position and size for each ticker
        self.positions = {ticker: 0 for ticker in tickers}
        self.sizes = {ticker: 0 for ticker in tickers}
        
    def push(self, ticker: str, state: np.ndarray, action: np.ndarray, 
             reward: float, next_state: np.ndarray, done: bool) -> None:
        """
        Store a transition for a specific ticker.
        
        Args:
            ticker: Ticker symbol
            state: Current state observation [state_dim]
            action: Action taken [n_tickers]
            reward: Reward received
            next_state: Next state observation [state_dim]
            done: Whether episode ended
        """
        if ticker not in self.buffers:
            raise ValueError(f"Unknown ticker: {ticker}")
            
        buffer = self.buffers[ticker]
        pos = self.positions[ticker]
        
        # Store transition
        buffer['states'][pos] = state
        buffer['actions'][pos] = action.flatten()  # Ensure action is flattened
        buffer['rewards'][pos] = reward
        buffer['next_states'][pos] = next_state
        buffer['dones'][pos] = done
        
        # Update position and size for this ticker
        self.positions[ticker] = (pos + 1) % self.capacity
        self.sizes[ticker] = min(self.sizes[ticker] + 1, self.capacity)

        
    def sample(self, batch_size: int, ticker: str = None) -> Tuple[torch.Tensor, ...]:
        """
        Sample a batch of transitions, either from a specific ticker or across all tickers.
        
        Args:
            batch_size (int): Number of transitions to sample
            ticker (str, optional): If provided, sample only from this ticker
            
        Returns:
            tuple: (states, actions, rewards, next_states, dones) as torch tensors
        """
        if ticker is not None:
            # Sample from specific ticker
            if ticker not in self.buffers:
                raise ValueError(f"Unknown ticker: {ticker}")
                
            buffer = self.buffers[ticker]
            size = self.sizes[ticker]
            indices = np.random.randint(0, size, size=batch_size)
            
            return (
                torch.FloatTensor(buffer['states'][indices]),
                torch.FloatTensor(buffer['actions'][indices]),
                torch.FloatTensor(buffer['rewards'][indices]),
                torch.FloatTensor(buffer['next_states'][indices]),
                torch.FloatTensor(buffer['dones'][indices])
            )
        else:
            # Sample across all tickers
            samples_per_ticker = batch_size // self.n_tickers
            remainder = batch_size % self.n_tickers
            
            states, actions, rewards, next_states, dones = [], [], [], [], []
            
            for i, ticker in enumerate(self.tickers):
                # Add one more sample to some tickers if batch_size doesn't divide evenly
                ticker_samples = samples_per_ticker + (1 if i < remainder else 0)
                
                if ticker_samples > 0:
                    buffer = self.buffers[ticker]
                    size = self.sizes[ticker]
                    indices = np.random.randint(0, size, size=ticker_samples)
                    
                    states.append(buffer['states'][indices])
                    actions.append(buffer['actions'][indices])
                    rewards.append(buffer['rewards'][indices])
                    next_states.append(buffer['next_states'][indices])
                    dones.append(buffer['dones'][indices])
            
            # Concatenate samples from all tickers
            return (
                torch.FloatTensor(np.concatenate(states)),
                torch.FloatTensor(np.concatenate(actions)),
                torch.FloatTensor(np.concatenate(rewards)),
                torch.FloatTensor(np.concatenate(next_states)),
                torch.FloatTensor(np.concatenate(dones))
            )
    
    def can_sample(self, batch_size: int, ticker: str = None) -> bool:
        """
        Check if enough transitions are stored to sample batch_size elements.
        
        Args:
            batch_size (int): Desired batch size
            ticker (str, optional): If provided, check only this ticker
            
        Returns:
            bool: True if enough transitions are stored
        """
        if ticker is not None:
            return self.sizes[ticker] >= batch_size
        return all(size >= batch_size // self.n_tickers for size in self.sizes.values())
    
    def __len__(self) -> Dict[str, int]:
        """Return current size of each ticker's buffer"""
        return self.sizes
    
    def clear(self):
        """Clear the replay buffer"""
        for ticker in self.buffers:
            self.buffers[ticker] = {
                'state': [],
                'action': [],
                'reward': [],
                'next_state': [],
                'done': []
            }
        gc.collect()

# Example usage and testing
def test_multi_ticker_buffer():
    # Example dimensions
    state_dim = 24 * 60  # 24 features with window size 60
    action_dim = 1       # Single continuous action per ticker
    tickers = ['LKOH', 'CHMF', 'SBER']  # Example tickers
    
    # Create buffer
    buffer = MultiTickerReplayBuffer(
        capacity=1000,
        state_dim=state_dim,
        action_dim=action_dim,
        tickers=tickers
    )
    
    # Test storing transitions for each ticker
    for ticker in tickers:
        for i in range(5):
            state = np.random.randn(state_dim)
            action = np.random.randn(action_dim)
            reward = np.random.rand()
            next_state = np.random.randn(state_dim)
            done = bool(np.random.randint(0, 2))
            
            buffer.push(ticker, state, action, reward, next_state, done)
    
    print("Buffer sizes after pushes:")
    for ticker, size in buffer.sizes.items():
        print(f"{ticker}: {size}")
    
    # Test sampling from specific ticker
    if buffer.can_sample(3, 'LKOH'):
        states, actions, rewards, next_states, dones = buffer.sample(3, 'LKOH')
        print(f"\nSample shapes for LKOH:")
        print(f"States: {states.shape}")
        print(f"Actions: {actions.shape}")
        print(f"Rewards: {rewards.shape}")
        print(f"Next states: {next_states.shape}")
        print(f"Dones: {dones.shape}")
    
    # Test sampling across all tickers
    if buffer.can_sample(9):
        states, actions, rewards, next_states, dones = buffer.sample(9)
        print(f"\nSample shapes for all tickers (9 samples):")
        print(f"States: {states.shape}")
        print(f"Actions: {actions.shape}")
        print(f"Rewards: {rewards.shape}")
        print(f"Next states: {next_states.shape}")
        print(f"Dones: {dones.shape}")
    
    return buffer

# Run test
buffer = test_multi_ticker_buffer()

Buffer sizes after pushes:
LKOH: 5
CHMF: 5
SBER: 5

Sample shapes for LKOH:
States: torch.Size([3, 1440])
Actions: torch.Size([3, 3])
Rewards: torch.Size([3])
Next states: torch.Size([3, 1440])
Dones: torch.Size([3])

Sample shapes for all tickers (9 samples):
States: torch.Size([9, 1440])
Actions: torch.Size([9, 3])
Rewards: torch.Size([9])
Next states: torch.Size([9, 1440])
Dones: torch.Size([9])


## 3.0 Actor critic networks

In [37]:
class Actor(nn.Module):
    """
    Actor network for SAC, outputs mean and log_std of action distribution.
    Uses a shared base network followed by ticker-specific heads for better
    feature extraction across multiple assets.
    """
    def __init__(self, state_dim: int, action_dim: int, n_tickers: int,
                 hidden_dim: int = 256, log_std_min: float = -20,
                 log_std_max: float = 2):
        """
        Initialize actor network.
        
        Args:
            state_dim: Dimension of state space (24 features * window_size)
            action_dim: Dimension of action space per ticker (1)
            n_tickers: Number of tickers being traded
            hidden_dim: Size of hidden layers
            log_std_min: Minimum log standard deviation
            log_std_max: Maximum log standard deviation
        """
        super(Actor, self).__init__()
        
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max
        self.n_tickers = n_tickers
        
        # Shared feature extraction layers
        self.shared_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),  # Normalization for better training stability
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # Ticker-specific networks for mean and log_std
        self.mean_nets = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.LayerNorm(hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, action_dim)
            ) for _ in range(n_tickers)
        ])
        
        self.log_std_nets = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.LayerNorm(hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, action_dim)
            ) for _ in range(n_tickers)
        ])
        
    def forward(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through the network.
        
        Args:
            state: Batch of states [batch_size, state_dim]
            
        Returns:
            tuple: (mean, log_std) for action distribution of each ticker
                  shapes: [batch_size, n_tickers, action_dim]
        """
        shared_features = self.shared_net(state)
        
        # Calculate mean and log_std for each ticker
        means = []
        log_stds = []
        
        for i in range(self.n_tickers):
            mean = self.mean_nets[i](shared_features)
            log_std = self.log_std_nets[i](shared_features)
            
            # Constrain log_std
            log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
            
            means.append(mean)
            log_stds.append(log_std)
        
        # Stack results for all tickers
        means = torch.stack(means, dim=1)      # [batch_size, n_tickers, action_dim]
        log_stds = torch.stack(log_stds, dim=1)  # [batch_size, n_tickers, action_dim]
        
        return means, log_stds

class Critic(nn.Module):
    """
    Critic network for SAC, estimates Q-values for state-action pairs.
    Uses a shared network for cross-asset feature extraction followed by
    ticker-specific Q-value heads.
    """
    def __init__(self, state_dim: int, action_dim: int, n_tickers: int,
                 hidden_dim: int = 256):
        """
        Initialize critic network.
        
        Args:
            state_dim: Dimension of state space (24 features * window_size)
            action_dim: Dimension of action space per ticker (1)
            n_tickers: Number of tickers being traded
            hidden_dim: Size of hidden layers
        """
        super(Critic, self).__init__()
        
        self.n_tickers = n_tickers
        
        # Shared feature extraction for state
        self.shared_state_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # Ticker-specific Q-networks
        self.q_nets = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim + action_dim * n_tickers, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.LayerNorm(hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1)
            ) for _ in range(n_tickers)
        ])
        
    def forward(self, state: torch.Tensor, 
                action: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the network.
        
        Args:
            state: Batch of states [batch_size, state_dim]
            action: Batch of actions [batch_size, n_tickers, action_dim]
            
        Returns:
            Q-values for each ticker [batch_size, n_tickers]
        """
        shared_features = self.shared_state_net(state)
        
        # Flatten actions for concatenation
        batch_size = state.shape[0]
        flat_actions = action.view(batch_size, -1)  # [batch_size, n_tickers * action_dim]
        
        # Concatenate shared features with all actions
        combined = torch.cat([shared_features, flat_actions], dim=1)
        
        # Calculate Q-values for each ticker
        q_values = []
        for i in range(self.n_tickers):
            q = self.q_nets[i](combined)
            q_values.append(q)
        
        # Stack Q-values for all tickers
        q_values = torch.cat(q_values, dim=1)  # [batch_size, n_tickers]
        
        return q_values

def test_networks():
    # Example parameters
    state_dim = 24 * 60  # 24 features with window size 60
    action_dim = 1       # Single continuous action per ticker
    n_tickers = 3       # Number of tickers
    batch_size = 32     # Batch size for testing
    
    # Create networks
    actor = Actor(state_dim, action_dim, n_tickers)
    critic = Critic(state_dim, action_dim, n_tickers)
    
    # Create example batch
    states = torch.randn(batch_size, state_dim)
    actions = torch.randn(batch_size, n_tickers, action_dim)
    
    # Test actor
    means, log_stds = actor(states)
    print("\nActor Output Shapes:")
    print(f"Means: {means.shape}")
    print(f"Log STDs: {log_stds.shape}")
    
    # Test critic
    q_values = critic(states, actions)
    print("\nCritic Output Shape:")
    print(f"Q-Values: {q_values.shape}")
    
    return actor, critic

# Run test
actor, critic = test_networks()


Actor Output Shapes:
Means: torch.Size([32, 3, 1])
Log STDs: torch.Size([32, 3, 1])

Critic Output Shape:
Q-Values: torch.Size([32, 3])


## 4.0 MultiTicker Trading Environment

In [30]:
class MultiTickerTradingEnv(gym.Env):
    def __init__(self, 
                 df: pd.DataFrame, 
                 window_size: int = 60,
                 transaction_cost: float = 0.0001,
                 initial_balance: float = 10000.0):
        super().__init__()
        
        # Process tickers
        self.tickers = sorted(df['tic'].unique())
        
        # Create dictionary of DataFrames for faster access
        self.ticker_dfs = {
            ticker: df[df['tic'] == ticker].copy() 
            for ticker in self.tickers
        }
        
        # Store total length for batch processing
        self.total_steps = min(len(df) for df in self.ticker_dfs.values())
        
        self.window_size = window_size
        self.transaction_cost = transaction_cost
        self.initial_balance = initial_balance
        
        # Define spaces
        self.action_space = gym.spaces.Box(
            low=-1, 
            high=1, 
            shape=(len(self.tickers),),
            dtype=np.float32
        )
        
        # Reduce observation space to essential features
        n_features = 6  # OHLCV + position
        self.observation_space = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(len(self.tickers), n_features, window_size),
            dtype=np.float32
        )
        
        # Initialize data window
        self.start_idx = 0
        self.end_idx = self.total_steps
        
        # Initialize state variables
        self.reset()
        
    def set_data_window(self, start_idx: int, end_idx: int):
        """Set the current data window for batch processing"""
        self.start_idx = max(self.window_size, start_idx)
        self.end_idx = min(self.total_steps, end_idx)
        self.current_step = self.start_idx
        
    def _get_observation(self) -> np.ndarray:
        observations = []
        
        for ticker in self.tickers:
            df = self.ticker_dfs[ticker]
            current_idx = self.current_step
            window_data = df.iloc[current_idx - self.window_size:current_idx]
            
            # Extract features
            features = np.column_stack([
                window_data['Open'].values,
                window_data['High'].values,
                window_data['Low'].values,
                window_data['Close'].values,
                window_data['Volume'].values,
                np.full(self.window_size, self.positions.get(ticker, 0))  # Current position
            ])
            
            observations.append(features.T)  # Transpose to get (features, window_size)
            
        return np.array(observations, dtype=np.float32)
    
    def reset(self):
        self.current_step = self.start_idx + self.window_size
        self.positions = {ticker: 0.0 for ticker in self.tickers}
        self.portfolio_value = self.initial_balance
        self.returns = []
        
        return self._get_observation()
    
    def step(self, actions):
        # Limit position sizes
        actions = np.clip(actions, -0.2, 0.2)  # Max 20% position size
        
        if self.current_step >= self.end_idx:
            return self._get_observation(), 0, True, {
                'portfolio_value': self.portfolio_value,
                'sharpe_ratio': self._calculate_sharpe_ratio(),
                'positions': self.positions
            }
        
        # Calculate returns and costs
        total_return = 0
        total_cost = 0
        
        for idx, ticker in enumerate(self.tickers):
            df = self.ticker_dfs[ticker]
            current_price = df.iloc[self.current_step]['Close']
            next_price = df.iloc[self.current_step + 1]['Close']
            
            # Calculate position change and cost
            old_position = self.positions[ticker]
            new_position = actions[idx]
            position_change = abs(new_position - old_position)
            transaction_cost = position_change * self.transaction_cost
            
            # Calculate return
            price_change = (next_price - current_price) / current_price
            position_return = new_position * price_change
            
            # Update totals
            total_return += position_return
            total_cost += transaction_cost
            
            # Update position
            self.positions[ticker] = new_position
        
        # Update portfolio value
        reward = total_return - total_cost
        self.portfolio_value *= (1 + reward)
        self.returns.append(reward)
        
        # Move to next step
        self.current_step += 1
        done = self.current_step >= self.end_idx
        
        return self._get_observation(), reward, done, {
            'portfolio_value': self.portfolio_value,
            'sharpe_ratio': self._calculate_sharpe_ratio(),
            'positions': self.positions
        }
    
    def _calculate_sharpe_ratio(self):
        if len(self.returns) < 2:
            return 0.0
        return np.mean(self.returns) / (np.std(self.returns) + 1e-8) * np.sqrt(252 * 1440)
    
    def clear(self):
        """Clear any cached data"""
        gc.collect()

def test_trading_env():
    """
    Test function for MultiTickerTradingEnv that verifies:
    1. Environment initialization
    2. Observation space and shape
    3. Action space and effects
    4. Reward calculation
    5. Portfolio value updates
    6. Data window functionality
    """
    print("Initializing environment...")
    env = MultiTickerTradingEnv(
        df=df,
        window_size=60,
        transaction_cost=0.001,
        initial_balance=10000.0
    )
    
    print(f"\nEnvironment Info:")
    print(f"Number of tickers: {len(env.tickers)}")
    print(f"Total timesteps: {env.total_steps}")
    print(f"Initial balance: ${env.initial_balance:,.2f}")
    
    # Test data window setting
    test_window = 1000
    env.set_data_window(0, test_window)
    print(f"\nSetting test window of {test_window} steps")
    
    # Test reset and initial observation
    print("\nTesting reset and initial observation...")
    initial_obs = env.reset()
    print(f"Observation shape: {initial_obs.shape}")
    print(f"Expected shape: (n_tickers, n_features, window_size) = "
          f"({len(env.tickers)}, 6, {env.window_size})")
    
    # Run a few test episodes
    n_test_steps = 5
    print(f"\nRunning {n_test_steps} test steps...")
    
    portfolio_history = [env.portfolio_value]
    position_history = []
    
    for step in range(n_test_steps):
        # Generate test actions (alternating between long and short positions)
        actions = np.random.uniform(-0.1, 0.1, len(env.tickers))
        # Take step
        obs, reward, done, info = env.step(actions)
        
        print(f"\nStep {step + 1}:")
        print(f"Actions taken: {actions.round(3)}")
        print(f"Reward: {reward:.6f}")
        print(f"Portfolio Value: ${info['portfolio_value']:,.2f}")
        print(f"Sharpe Ratio: {info['sharpe_ratio']:.4f}")
        
        # Store history
        portfolio_history.append(info['portfolio_value'])
        position_history.append(info['positions'])
        
        if done:
            print("\nEpisode finished early")
            break
    
    # Calculate and display summary statistics
    returns = np.diff(portfolio_history) / portfolio_history[:-1]
    
    print("\nTest Summary:")
    print(f"Total Return: {((portfolio_history[-1] / portfolio_history[0]) - 1) * 100:.2f}%")
    print(f"Average Step Return: {np.mean(returns) * 100:.4f}%")
    print(f"Return Volatility: {np.std(returns) * 100:.4f}%")
    
    # Test memory cleanup
    print("\nTesting memory cleanup...")
    env.clear()
    print("Environment cleanup completed")
    
    return env

# Run the test
if __name__ == "__main__":
    test_env = test_trading_env()

Initializing environment...

Environment Info:
Number of tickers: 49
Total timesteps: 131106
Initial balance: $10,000.00

Setting test window of 1000 steps

Testing reset and initial observation...
Observation shape: (49, 6, 60)
Expected shape: (n_tickers, n_features, window_size) = (49, 6, 60)

Running 5 test steps...

Step 1:
Actions taken: [ 0.024  0.08   0.083 -0.05   0.053 -0.035 -0.003  0.061 -0.002 -0.056
  0.012 -0.084 -0.036  0.098  0.095  0.043 -0.042 -0.     0.041 -0.028
  0.004 -0.016 -0.033 -0.057  0.066  0.084 -0.053  0.084  0.005  0.056
  0.008 -0.017 -0.057 -0.069  0.098  0.053  0.028 -0.071 -0.017  0.083
 -0.006 -0.065  0.004  0.039 -0.007 -0.01   0.022 -0.099  0.028]
Reward: -0.001954
Portfolio Value: $9,980.46
Sharpe Ratio: 0.0000

Step 2:
Actions taken: [ 0.006  0.042  0.022  0.049 -0.08   0.067 -0.022 -0.097  0.098 -0.003
  0.01   0.004 -0.075  0.064  0.001  0.097  0.095 -0.023  0.012  0.028
  0.073  0.085 -0.079  0.061  0.063  0.023 -0.057  0.057  0.082  0.02
 -0.

## 5.0 SACAgent

In [36]:
class SACAgent:
    def __init__(self,
                 state_dim: int,
                 action_dim: int,
                 n_tickers: int,
                 hidden_dim: int = 256,
                 learning_rate: float = 3e-4,
                 alpha: float = 0.2):
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Initialize networks
        self.actor = Actor(state_dim, action_dim * n_tickers, hidden_dim).to(self.device)
        self.critic_1 = Critic(state_dim + action_dim * n_tickers, hidden_dim).to(self.device)
        self.critic_2 = Critic(state_dim + action_dim * n_tickers, hidden_dim).to(self.device)
        self.critic_1_target = Critic(state_dim + action_dim * n_tickers, hidden_dim).to(self.device)
        self.critic_2_target = Critic(state_dim + action_dim * n_tickers, hidden_dim).to(self.device)
        
        # Copy parameters to target networks
        self.critic_1_target.load_state_dict(self.critic_1.state_dict())
        self.critic_2_target.load_state_dict(self.critic_2.state_dict())
        
        # Initialize optimizers
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=learning_rate)
        self.critic_1_optimizer = optim.Adam(self.critic_1.parameters(), lr=learning_rate)
        self.critic_2_optimizer = optim.Adam(self.critic_2.parameters(), lr=learning_rate)
        
        # Initialize temperature parameter alpha
        self.target_entropy = -action_dim * n_tickers  # Target entropy is -|A|
        self.log_alpha = torch.zeros(1, requires_grad=True, device=self.device)
        self.alpha = alpha
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=learning_rate)
        
        self.n_tickers = n_tickers
        self.action_dim = action_dim
        
    def select_action(self, state: np.ndarray) -> np.ndarray:
        """Select action for given state with noise for exploration"""
        with torch.no_grad():
            state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            mean, log_std = self.actor(state)
            std = log_std.exp()
            normal = Normal(mean, std)
            z = normal.sample()
            action = torch.tanh(z)
            action = action.cpu().numpy()[0]
        return action.reshape(self.n_tickers, self.action_dim)
    
    def update_parameters(self, 
                         replay_buffer: MultiTickerReplayBuffer, 
                         batch_size: int,
                         gamma: float = 0.99,
                         tau: float = 0.005) -> dict:
        """Update agent parameters using batch of experience"""
        # Sample batch from replay buffer
        state_batch, action_batch, reward_batch, next_state_batch, done_batch = \
            replay_buffer.sample(batch_size)
        
        # Convert to tensors
        state_batch = torch.FloatTensor(state_batch).to(self.device)
        action_batch = torch.FloatTensor(action_batch).to(self.device)
        reward_batch = torch.FloatTensor(reward_batch).to(self.device)
        next_state_batch = torch.FloatTensor(next_state_batch).to(self.device)
        done_batch = torch.FloatTensor(done_batch).to(self.device)
        
        # Reshape action batch if necessary
        if action_batch.dim() == 3:  # If shape is [batch_size, n_tickers, action_dim]
            action_batch = action_batch.reshape(batch_size, -1)
        
        # Update critics
        with torch.no_grad():
            next_state_action, next_state_log_pi = self._compute_policy_values(next_state_batch)
            
            # Target Q-values
            qf1_next_target = self.critic_1_target(next_state_batch, next_state_action)
            qf2_next_target = self.critic_2_target(next_state_batch, next_state_action)
            min_qf_next_target = torch.min(qf1_next_target, qf2_next_target) - self.alpha * next_state_log_pi
            next_q_value = reward_batch + (1 - done_batch) * gamma * min_qf_next_target
        
        # Current Q-values
        qf1 = self.critic_1(state_batch, action_batch)
        qf2 = self.critic_2(state_batch, action_batch)
        
        # Compute critic losses
        qf1_loss = F.mse_loss(qf1, next_q_value)
        qf2_loss = F.mse_loss(qf2, next_q_value)
        
        # Update critics
        self.critic_1_optimizer.zero_grad()
        qf1_loss.backward()
        self.critic_1_optimizer.step()
        
        self.critic_2_optimizer.zero_grad()
        qf2_loss.backward()
        self.critic_2_optimizer.step()
        
        # Update actor
        pi, log_pi = self._compute_policy_values(state_batch)
        qf1_pi = self.critic_1(state_batch, pi)
        qf2_pi = self.critic_2(state_batch, pi)
        min_qf_pi = torch.min(qf1_pi, qf2_pi)
        
        policy_loss = ((self.alpha * log_pi) - min_qf_pi).mean()
        
        self.actor_optimizer.zero_grad()
        policy_loss.backward()
        self.actor_optimizer.step()
        
        # Update temperature parameter alpha
        alpha_loss = -(self.log_alpha * (log_pi + self.target_entropy).detach()).mean()
        
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()
        
        self.alpha = self.log_alpha.exp()
        
        # Update target networks
        self._soft_update(self.critic_1, self.critic_1_target, tau)
        self._soft_update(self.critic_2, self.critic_2_target, tau)
        
        return {
            'actor_loss': policy_loss.item(),
            'critic_1_loss': qf1_loss.item(),
            'critic_2_loss': qf2_loss.item(),
            'alpha': self.alpha.item()
        }
    
    def _compute_policy_values(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Compute policy actions and log probabilities for given state"""
        mean, log_std = self.actor(state)
        std = log_std.exp()
        normal = Normal(mean, std)
        x_t = normal.rsample()  # Reparameterization trick
        action = torch.tanh(x_t)
        
        log_prob = normal.log_prob(x_t)
        # Enforcing action bounds
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(-1, keepdim=True)
        
        return action, log_prob
    
    def _soft_update(self, local_model: nn.Module, target_model: nn.Module, tau: float):
        """Soft update target network parameters"""
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(tau * local_param.data + (1.0 - tau) * target_param.data)
            
    def state_dict(self) -> dict:
        """Return state dict for checkpointing"""
        return {
            'actor': self.actor.state_dict(),
            'critic_1': self.critic_1.state_dict(),
            'critic_2': self.critic_2.state_dict(),
            'critic_1_target': self.critic_1_target.state_dict(),
            'critic_2_target': self.critic_2_target.state_dict(),
            'actor_optimizer': self.actor_optimizer.state_dict(),
            'critic_1_optimizer': self.critic_1_optimizer.state_dict(),
            'critic_2_optimizer': self.critic_2_optimizer.state_dict(),
            'log_alpha': self.log_alpha,
            'alpha_optimizer': self.alpha_optimizer.state_dict()
        }
    
    def load_state_dict(self, state_dict: dict):
        """Load state dict from checkpoint"""
        self.actor.load_state_dict(state_dict['actor'])
        self.critic_1.load_state_dict(state_dict['critic_1'])
        self.critic_2.load_state_dict(state_dict['critic_2'])
        self.critic_1_target.load_state_dict(state_dict['critic_1_target'])
        self.critic_2_target.load_state_dict(state_dict['critic_2_target'])
        self.actor_optimizer.load_state_dict(state_dict['actor_optimizer'])
        self.critic_1_optimizer.load_state_dict(state_dict['critic_1_optimizer'])
        self.critic_2_optimizer.load_state_dict(state_dict['critic_2_optimizer'])
        self.log_alpha = state_dict['log_alpha']
        self.alpha_optimizer.load_state_dict(state_dict['alpha_optimizer'])


def test_sac_agent():
    """
    Test function to verify SACAgent implementation:
    1. Agent initialization
    2. Action selection
    3. Parameter updates
    4. State dict saving/loading
    """
    print("Initializing test environment...")
    env = MultiTickerTradingEnv(
        df=df,
        window_size=60,
        transaction_cost=0.0001  # Lower transaction cost for testing
    )
    
    # Get dimensions from environment
    n_tickers = len(env.tickers)
    state_dim = env.observation_space.shape[0]
    action_dim = 1  # Position size per ticker
    
    print(f"\nEnvironment Info:")
    print(f"Number of tickers: {n_tickers}")
    print(f"State dimension: {state_dim}")
    print(f"Action dimension per ticker: {action_dim}")
    
    print("\nInitializing SAC agent...")
    agent = SACAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        n_tickers=n_tickers,
        hidden_dim=64,  # Smaller network for testing
        learning_rate=3e-4,
        alpha=0.2
    )
    
    print("\nInitializing replay buffer...")
    replay_buffer = MultiTickerReplayBuffer(
        capacity=10000,  # Smaller buffer for testing
        state_dim=state_dim,
        action_dim=action_dim,
        tickers=env.tickers
    )
    
    print("\nTesting action selection...")
    state = env.reset()
    action = agent.select_action(state)
    print(f"Action shape: {action.shape}")
    print(f"Sample action values:\n{action.round(3)}")
    
    print("\nCollecting some experience...")
    n_steps = 100
    for step in range(n_steps):
        action = agent.select_action(state)
        next_state, reward, done, info = env.step(action)
        
        # Store transition in replay buffer
        for ticker_idx, ticker in enumerate(env.tickers):
            replay_buffer.push(
                ticker,
                state,
                action[ticker_idx],
                reward,
                next_state,
                done
            )
        
        state = next_state
        if done:
            state = env.reset()
    
    print(f"\nCollected {n_steps} steps of experience")
    
    print("\nTesting parameter update...")
    if replay_buffer.can_sample(64):
        update_info = agent.update_parameters(replay_buffer, batch_size=64)
        print("\nUpdate info:")
        print(f"Actor loss: {update_info['actor_loss']:.4f}")
        print(f"Critic 1 loss: {update_info['critic_1_loss']:.4f}")
        print(f"Critic 2 loss: {update_info['critic_2_loss']:.4f}")
        print(f"Alpha: {update_info['alpha']:.4f}")
    else:
        print("Not enough samples in buffer for update")
    
    print("\nTesting state dict saving/loading...")
    # Save state dict
    state_dict = agent.state_dict()
    print("State dict keys:", state_dict.keys())
    
    # Create new agent and load state dict
    new_agent = SACAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        n_tickers=n_tickers,
        hidden_dim=64
    )
    new_agent.load_state_dict(state_dict)
    
    print("\nComparing actions between original and loaded agents...")
    test_state = env.reset()
    original_action = agent.select_action(test_state)
    loaded_action = new_agent.select_action(test_state)
    action_diff = np.abs(original_action - loaded_action).mean()
    print(f"Mean action difference: {action_diff:.6f}")
    
    print("\nTest completed successfully!")
    return agent, env, replay_buffer

# Run the test
if __name__ == "__main__":
    agent, env, replay_buffer = test_sac_agent()


Initializing test environment...

Environment Info:
Number of tickers: 49
State dimension: 49
Action dimension per ticker: 1

Initializing SAC agent...


TypeError: Critic.__init__() missing 1 required positional argument: 'n_tickers'

In [33]:
class SACTrainer:
    def __init__(self, 
                 agent: SACAgent,
                 env: MultiTickerTradingEnv,
                 replay_buffer: MultiTickerReplayBuffer,
                 log_dir: str = "logs/sac_training",
                 batch_size: int = 64,
                 training_window: int = 10000,
                 max_memory_usage: float = 0.8):  # Added max_memory_usage parameter
        
        self.agent = agent
        self.env = env
        self.replay_buffer = replay_buffer
        self.batch_size = batch_size
        self.training_window = training_window
        self.max_memory_usage = max_memory_usage
        
        # Create log directory with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_dir = Path(log_dir) / timestamp
        self.log_dir.mkdir(parents=True, exist_ok=True)
        
        # Initialize logger
        self.logger = self._setup_logger()
        
        # Training metrics
        self.metrics = {
            'episode_rewards': [],
            'sharpe_ratios': [],
            'portfolio_values': [],
            'actor_losses': [],
            'critic_1_losses': [],
            'critic_2_losses': [],
            'alphas': [],
            'positions': {ticker: [] for ticker in env.tickers}
        }

    def _check_memory_usage(self) -> bool:
        """
        Monitor memory usage and clear cache if needed.
        Returns True if memory usage is OK, False if too high.
        """
        import psutil
        
        memory_percent = psutil.Process().memory_percent()
        if memory_percent > self.max_memory_usage:
            self.logger.warning(f"Memory usage high: {memory_percent:.1f}%")
            gc.collect()
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            
            # Check if memory is still too high after cleanup
            memory_percent = psutil.Process().memory_percent()
            if memory_percent > self.max_memory_usage:
                return False
        return True

    def _save_checkpoint(self, episode: int, is_best: bool = False):
        """
        Save model checkpoint and training state.
        
        Args:
            episode: Current episode number
            is_best: Whether this is the best model so far
        """
        checkpoint = {
            'episode': episode,
            'agent_state_dict': self.agent.state_dict(),
            'metrics': self.metrics
        }
        
        if is_best:
            path = self.log_dir / 'best_model.pth'
        else:
            path = self.log_dir / f'checkpoint_ep{episode}.pth'
            
        torch.save(checkpoint, path)
        self.logger.info(f"Saved checkpoint to {path}")

    def _update_metrics(self, update_info: dict):
        """Update training metrics with new values"""
        self.metrics['actor_losses'].append(update_info.get('actor_loss', 0))
        self.metrics['critic_1_losses'].append(update_info.get('critic_1_loss', 0))
        self.metrics['critic_2_losses'].append(update_info.get('critic_2_loss', 0))
        self.metrics['alphas'].append(update_info.get('alpha', 0))

    def _log_episode(self, episode: int, reward: float, info: dict, steps: int):
        """Log episode results"""
        self.metrics['episode_rewards'].append(reward)
        self.metrics['sharpe_ratios'].append(info['sharpe_ratio'])
        self.metrics['portfolio_values'].append(info['portfolio_value'])
        
        for ticker, position in info['positions'].items():
            self.metrics['positions'][ticker].append(position)
        
        self.logger.info(
            f"\nEpisode {episode + 1}"
            f"\nSteps: {steps}"
            f"\nReward: {reward:.4f}"
            f"\nSharpe Ratio: {info['sharpe_ratio']:.4f}"
            f"\nPortfolio Value: {info['portfolio_value']:.4f}"
        )

    def _plot_metrics(self):
        """Plot training metrics (placeholder for now)"""
        # TODO: Implement plotting functionality
        pass

    def _save_metrics(self):
        """Save metrics to file"""
        metrics_path = self.log_dir / 'metrics.json'
        with open(metrics_path, 'w') as f:
            json.dump(self.metrics, f, indent=4)
        self.logger.info(f"Saved metrics to {metrics_path}")

    def _setup_logger(self) -> logging.Logger:
        """Configure and return logger"""
        logger = logging.getLogger(f'SAC_Training_{datetime.now().strftime("%Y%m%d_%H%M%S")}')
        logger.setLevel(logging.INFO)
        
        # Remove any existing handlers
        if logger.hasHandlers():
            logger.handlers.clear()
        
        # File handler
        fh = logging.FileHandler(self.log_dir / 'training.log')
        fh.setLevel(logging.INFO)
        
        # Console handler
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        
        # Formatter
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        ch.setFormatter(formatter)
        
        logger.addHandler(fh)
        logger.addHandler(ch)
        
        return logger
    
    def train(self, 
              num_episodes: int,
              warmup_steps: int = 1000,
              update_interval: int = 1,
              eval_interval: int = 10,
              save_interval: int = 100):
        """
        Train the agent using batch processing.
        """
        self.logger.info(f"Starting training for {num_episodes} episodes")
        self.logger.info(f"Batch size: {self.batch_size}, Warmup steps: {warmup_steps}")
        
        # Get total timesteps from environment
        total_timesteps = self.env.total_steps - self.env.window_size  # Use total_steps instead of df length
        num_batches = total_timesteps // self.training_window
        
        total_steps = 0
        best_sharpe = float('-inf')
        episode = 0
        
        try:
            # Process data in batches
            for batch_idx in range(num_batches):
                self.logger.info(f"Processing batch {batch_idx + 1}/{num_batches}")
                
                # Set environment data range for current batch
                start_idx = batch_idx * self.training_window
                end_idx = start_idx + self.training_window
                self.env.set_data_window(start_idx, end_idx)
                
                # Train on current batch
                batch_episodes = num_episodes // num_batches
                for local_episode in range(batch_episodes):
                    if not self._check_memory_usage():
                        self.logger.warning("Memory usage too high, saving checkpoint and exiting...")
                        self._save_checkpoint(episode, is_best=False)
                        return self.metrics
                    
                    state = self.env.reset()
                    episode_reward = 0
                    episode_steps = 0
                    done = False
                    
                    while not done:
                        # Select action
                        if total_steps < warmup_steps:
                            action = self.env.action_space.sample()
                        else:
                            action = self.agent.select_action(state)
                        
                        # Take step in environment
                        next_state, reward, done, info = self.env.step(action)
                        
                        # Store transition in replay buffer
                        for ticker_idx, ticker in enumerate(self.env.tickers):
                            self.replay_buffer.push(
                                ticker,
                                state,
                                action[ticker_idx],
                                reward,
                                next_state,
                                done
                            )
                        
                        state = next_state
                        episode_reward += reward
                        episode_steps += 1
                        total_steps += 1
                        
                        # Update networks
                        if total_steps > warmup_steps and total_steps % update_interval == 0:
                            if self.replay_buffer.can_sample(self.batch_size):
                                update_info = self.agent.update_parameters(
                                    self.replay_buffer,
                                    self.batch_size
                                )
                                self._update_metrics(update_info)
                        
                        # Clear memory periodically
                        if episode_steps % 100 == 0:
                            self._check_memory_usage()
                    
                    # Episode complete - log metrics
                    self._log_episode(episode, episode_reward, info, episode_steps)
                    
                    # Evaluate and save
                    if (episode + 1) % eval_interval == 0:
                        self._plot_metrics()
                        self._save_metrics()
                    
                    if (episode + 1) % save_interval == 0:
                        self._save_checkpoint(episode + 1)
                        
                        # Save best model based on Sharpe ratio
                        if info['sharpe_ratio'] > best_sharpe:
                            best_sharpe = info['sharpe_ratio']
                            self._save_checkpoint(episode + 1, is_best=True)
                    
                    episode += 1
                
                # Clear replay buffer between batches to free memory
                self.replay_buffer.clear()
                gc.collect()
                
        except Exception as e:
            self.logger.error(f"Training interrupted: {str(e)}")
            self._save_checkpoint(episode, is_best=False)
            raise
        
        self.logger.info("Training completed")
        return self.metrics

# Modified training function to use total_steps
def train_agent(num_episodes: int = 1000, training_window: int = 10000):
    env = MultiTickerTradingEnv(
        df=df,
        window_size=60
    )
    
    state_dim = env.observation_space.shape[0]
    action_dim = 1
    n_tickers = len(env.tickers)
    
    agent = SACAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        n_tickers=n_tickers,
        hidden_dim=256
    )
    
    replay_buffer = MultiTickerReplayBuffer(
        capacity=100_000,
        state_dim=state_dim,
        action_dim=action_dim,
        tickers=env.tickers
    )
    
    trainer = SACTrainer(
        agent=agent,
        env=env,
        replay_buffer=replay_buffer,
        training_window=training_window
    )
    
    try:
        metrics = trainer.train(
            num_episodes=num_episodes,
            warmup_steps=1000,
            update_interval=1,
            eval_interval=10,
            save_interval=100
        )
    except Exception as e:
        print(f"Training interrupted: {e}")
        metrics = trainer.metrics
    
    return trainer, metrics

# Run training with batch processing
trainer, metrics = train_agent(
    num_episodes=1000,
    training_window=10000  # Process 10,000 timesteps at a time
)

2025-03-23 19:59:44,236 - SAC_Training_20250323_195944 - INFO - Starting training for 1000 episodes
2025-03-23 19:59:44,237 - SAC_Training_20250323_195944 - INFO - Batch size: 64, Warmup steps: 1000
2025-03-23 19:59:44,240 - SAC_Training_20250323_195944 - INFO - Processing batch 1/13
2025-03-23 19:59:44,262 - SAC_Training_20250323_195944 - WARNING - Memory usage high: 93.7%
2025-03-23 19:59:46,784 - SAC_Training_20250323_195944 - WARNING - Memory usage too high, saving checkpoint and exiting...
2025-03-23 19:59:46,785 - SAC_Training_20250323_195944 - ERROR - Training interrupted: 'SACAgent' object has no attribute 'state_dict'


Training interrupted: 'SACAgent' object has no attribute 'state_dict'
